# Colab → real-GPU WebGPU smoke test for `remove-photo-bg`

The sandbox/CI have **no GPU**, so the WebGPU path of the real product (`detectWebGpu` → `ort.all.min.js` + jsep wasm → WebGPU inference) has never been exercised against the actual app. Google's official **"Web AI model testing in Google Colab"** recipe ([developer.chrome.com/docs/web-platform/webgpu/colab-headless](https://developer.chrome.com/docs/web-platform/webgpu/colab-headless)) runs headless Chrome with Vulkan/WebGPU on a free Colab **T4**. This notebook uses that recipe to open the app and test the GPU path for real.

**What this validates:**
- `navigator.gpu` + `requestAdapter()` succeed on the T4 (our gate).
- The host serves the WebGPU-capable ORT bundle (`ort.all.min.js` + `ort-wasm-simd-threaded.jsep.wasm`) — the app only *offers* WebGPU when this probe passes.
- A WebGPU session builds and runs, and the provider reported by the app is `webgpu`.
- No image pixels leave the page (every network request is logged and filtered).

**Cost:** none (free T4 tier). **Limits:** Colab session dies after ~12h; the GPU is not redirectable to your users' browsers — this is a *test* harness, not hosting.

## Setup

1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Fill in the **`PAGE_URL`** cell. For a private/local page (e.g. a `vite preview` running on the machine that owns the model), use `cloudflared tunnel --url http://localhost:4173` to get a temporary public HTTPS URL, or point at an already-deployed preview that serves the jsep bundle.
3. Run every cell top to bottom.

In [0]:
# chromedp=platform=linux one-shot Chrome driver (the recipe's approach)
!wget -q https://storage.googleapis.com/chrome-for-testing-public/132.0.6834.83/linux64/chromedriver-linux64.zip -O /tmp/cd.zip && unzip -o -q /tmp/cd.zip -d /tmp && ls /tmp/chromedriver-linux64/

In [0]:
# 1. Install headless Chrome (official recipe) + the small node driver deps
!git clone --depth 1 https://github.com/GoogleChromeLabs/headless-chrome-nvidia-t4-gpu-support.git /tmp/t4gpu 2>/dev/null || true
!chmod +x /tmp/t4gpu/scriptyMcScriptFace.sh && /tmp/t4gpu/scriptyMcScriptFace.sh 2>&1 | tail -5

# 2. Node + puppeteer-core + onnxruntime (sizes only) via npm cache-less install
!npm init -y >/dev/null 2>&1; npm install --no-save puppeteer-core onnxruntime-web >/dev/null 2>&1
!node -e "console.log('node', process.version, '| puppeteer-core ^', require('puppeteer-core/package.json').version, '| onnxruntime-web', require('onnxruntime-web/package.json').version)"

In [0]:
import os
import json

# REPLACE ME. Best source: `cloudflared tunnel --url http://localhost:4173` on
# the machine running `vite preview` with `npm run fetch-models -- --webgpu`
# baked in, so /wasm/ort.all.min.js + the jsep binary are served same-origin.
# The host MUST set COOP/COEP (vite preview does) and serve the jsep.
PAGE_URL = "https://<your-preview-url>"

OUT = "/content/remove-photo-bg-colab/colab-results"
os.makedirs(OUT, exist_ok=True)
print("PAGE_URL =", PAGE_URL)

In [0]:
%%writefile /content/remove-photo-bg-colab/colab-smoke.mjs
// GPU-enabled headless driver for the remove-photo-bg app.
// Mirrors scripts/run-smoke.mjs but: (a) runs in Colab's headless Chrome with
// the WebGPU flags, (b) asserts the provider is 'webgpu' (the point), and
// (c) logs the app's own device report + provider line. Everything stays local
// to the notebook; the only network egress is the page itself.
import puppeteer from 'puppeteer-core';

const PAGE_URL = process.env.PAGE_URL;
const CHROME = process.env.CHROME_BIN;
const OUT = process.env.OUT_DIR;
const MODEL = process.env.SMOKE_MODEL || 'c5_best';
const IMAGE = process.env.SMOKE_IMAGE || 'test-images/glass-wine.jpg';

const browser = await puppeteer.launch({
  executablePath: CHROME,
  headless: 'shell',
  protocolTimeout: 3600000,
  timeout: 3600000,
  // Official Colab T4 recipe: Vulkan + Software WebGPU + unsafe WebGPU.
  args: [
    '--no-sandbox',
    '--headless=new',
    '--use-angle=vulkan',
    '--enable-features=Vulkan',
    '--disable-vulkan-surface',
    '--enable-unsafe-webgpu',
    '--disable-dev-shm-usage',
  ],
});

const page = await browser.newPage();
const requests = [];
page.on('request', (req) => {
  if (req.url().startsWith('data:')) return;
  requests.push({ method: req.method(), url: req.url().slice(0, 200), type: req.resourceType() });
});
page.on('pageerror', (e) => console.error('[pageerror]', e.message));
page.on('console', (m) => { const t = m.text(); if (t.includes('log')) console.log('[console]', t.slice(0, 300)); });
page.on('dialog', (d) => d.dismiss());

const url = new URL(PAGE_URL);
url.searchParams.set('model', MODEL);
url.searchParams.set('provider', 'webgpu'); // force the GPU path
await page.goto(url.toString(), { waitUntil: 'load', timeout: 120000 });

const ready = await page.waitForFunction('window.smokeModelReady === true', { timeout: 600000 })
  .then(() => true)
  .catch(async () => {
    const st = await page.evaluate(() => ({
      stage: document.getElementById('stage')?.textContent,
      log: document.getElementById('log')?.textContent,
      provider: window.smokeProvider,
      model: window.smokeModelId,
    }));
    console.error('Model did not become ready:', JSON.stringify(st, null, 2));
    return false;
  });
if (!ready) { await browser.close(); process.exit(1); }

const info = await page.evaluate(() => ({
  provider: window.smokeProvider,
  model: window.smokeModelId,
  adapter: document.getElementById('deviceReport')?.textContent ?? '',
  runtimeOptions: Array.from(document.querySelectorAll('#modelSelect option'))
    .filter((o) => o.value.startsWith('runtime:'))
    .map((o) => o.value),
}));
console.log('READY provider=', info.provider, 'model=', info.model, 'runtimeOptions=', info.runtimeOptions.join(','));

if (info.provider !== 'webgpu') {
  console.error('EXPECTED provider webgpu, got', info.provider);
  await browser.close();
  process.exit(1);
}

const envInfo = await page.evaluate(() => ({
  crossOriginIsolated: globalThis.crossOriginIsolated === true,
  hasSharedArrayBuffer: typeof SharedArrayBuffer !== 'undefined',
}));
console.log('env', JSON.stringify(envInfo));

const runResult = await page.evaluate(async (img, n) => await window.runSmoke(img, n), IMAGE, 1);
console.log('RUN provider=', runResult.provider, 'model=', runResult.modelId, 'timings=', runResult.timings);

const providerLine = await page.evaluate(() => document.querySelector('#outputBox')?.textContent ?? '');
const report = await page.evaluate(() => document.getElementById('deviceReport')?.textContent ?? '');

import { mkdirSync, writeFileSync } from 'node:fs';
mkdirSync(OUT, { recursive: true });
writeFileSync(OUT + '/summary.json', JSON.stringify({
  url: PAGE_URL, provider: runResult.provider, model: runResult.modelId,
  timings: runResult.timings, env: envInfo, adapter: info.adapter,
  log: await page.evaluate(() => document.querySelector('#log')?.textContent),
}, null, 2));
writeFileSync(OUT + '/requests.json', JSON.stringify(requests, null, 2));
console.log('--- suspicious (fetch/xhr/api) ---');
for (const r of requests.filter((r) => r.type === 'fetch' || r.type === 'xhr' || r.url.includes('api')))
  console.log(' ', r.method, r.type, r.url);
console.log('--- device report ---\n' + report);

await browser.close();


In [0]:
# 3. Run the GPU smoke (this is the real test).
import os

os.environ['PAGE_URL'] = PAGE_URL
os.environ['CHROME_BIN'] = '/usr/bin/google-chrome-stable'
os.environ['OUT_DIR'] = OUT
os.environ['SMOKE_MODEL'] = 'c5_best'

!/content/remove-photo-bg-colab/.venv  # (guard)
!cd /content/remove-photo-bg-colab && node colab-smoke.mjs 2>&1 | tail -40

## Reading the result

- `READY provider=webgpu` → the T4 adapter passed the gate and the jsep bundle was served.
- `RUN provider=webgpu` → ORT built the session on the GPU and inference ran on it.
- Post the `--- device report ---` block (adapter, f16 support, provider line) back with your timing.
- Any `suspicious` fetch/xhr line is a deviation from the *no pixels leave the page* guarantee.

A *failure* to reach `webgpu` usually means one of: the host doesn't serve the jsep bundle (run `npm run fetch-models -- --webgpu` and rebuild), the page isn't cross-origin isolated (needs COOP/COEP), or Chrome's Vulkan/WebGPU flags didn't engage. The app reports a **provider-fallback** message in the log when a WebGPU attempt degrades to WASM — if you see it, capture the log.

## How this maps to the product

- **`navigator.gpu` + `requestAdapter()`** → `provider-gate.detectWebGpu`
- **host serves jsep** → `webGpuRuntimeHosted()` probe (`provider-gate.ts`)
- **`ort.all.min.js` vs `ort.wasm.min.js`** → chosen in `main.ts` via `?runtime=webgpu` on the worker URL
- **WebGPU session** → `inference-worker.js` candidates `['webgpu','wasm']`, fallback on failure

This notebook is what lets us close the product-validation gap without any GPU in our sandbox or a credit card.